# MV Validation — Overlapping-Window Group-Aware Label Split (Issue 3B)

This notebook implements the relatively less restrictive option in issue 3B.

> For the same `login_id + acct_nbr`, only samples whose 90-day windows overlap must stay in one split. Non-overlapping windows from the same employee-account pair may be assigned to different splits.

Overlap is transitive. If window A overlaps B and B overlaps C, all three windows form one indivisible overlap group even when A does not directly overlap C.

The notebook follows the issue 3A validation structure and uses minimal-change balanced greedy repair. Groups that already belong to one original split are kept unchanged. Only overlap groups that currently cross splits are reassigned, using a global loss over row counts, positive counts, moved rows, and overshoot. Multiple randomized group orders are attempted and the lowest-loss result is retained. It does not retrain a model or overwrite the source table.

In [ ]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

## 1. Configuration

The permanently saved LightGBM feature table is sufficient because it contains the employee, account, label, original split, and 90-day window boundaries.

In [ ]:
FEATURE_TABLE_PATH = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'ins_us_nms/v1/output/insider_us_nms_features_table_v2'
)

EMPLOYEE_COL = 'login_id'
ACCOUNT_COL = 'acct_nbr'
SAMPLE_DATE_COL = 'date'
WINDOW_START_COL = 'lookback_window_start'
WINDOW_END_COL = 'fraud_date'
TARGET_COL = 'label'
ORIGINAL_SPLIT_COL = 'label_split'
NEW_SPLIT_COL = 'new_label_split'
OVERLAP_GROUP_COL = 'overlap_group_id'

SPLIT_ORDER = ['train', 'val', 'test']
RANDOM_SEED = 42
N_ATTEMPTS = 10
ROW_WEIGHT = 1.0
POSITIVE_WEIGHT = 3.0
MOVE_WEIGHT = 0.5
OVERSHOOT_WEIGHT = 5.0
EPSILON = 1e-12

N_CV_FOLDS = 5
CV_FOLD_SEED = 43
FULL_FEATURE_OUTPUT_PATH = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'mrm/ins_us_nms/output/insider_us_nms_features_table_overlap_split_v1'
)

## 2. Read the permanent feature table

In [ ]:
features_df = (
    spark.read.format('delta').load(FEATURE_TABLE_PATH)
    .select(
        F.upper(F.col(EMPLOYEE_COL)).alias(EMPLOYEE_COL),
        F.col(ACCOUNT_COL).cast('string').alias(ACCOUNT_COL),
        F.to_date(SAMPLE_DATE_COL).alias(SAMPLE_DATE_COL),
        F.to_date(WINDOW_START_COL).alias(WINDOW_START_COL),
        F.to_date(WINDOW_END_COL).alias(WINDOW_END_COL),
        F.col(TARGET_COL).cast('int').alias(TARGET_COL),
        F.col(ORIGINAL_SPLIT_COL),
    )
    .cache()
)

required_cols = [
    EMPLOYEE_COL, ACCOUNT_COL, SAMPLE_DATE_COL, WINDOW_START_COL,
    WINDOW_END_COL, TARGET_COL, ORIGINAL_SPLIT_COL,
]
invalid_input_count = features_df.filter(
    (F.greatest(*[F.col(c).isNull().cast('int') for c in required_cols]) == 1)
    | (F.col(WINDOW_START_COL) > F.col(WINDOW_END_COL))
    | (~F.col(TARGET_COL).isin(0, 1))
    | (~F.col(ORIGINAL_SPLIT_COL).isin(SPLIT_ORDER))
).count()
assert invalid_input_count == 0, f'Found {invalid_input_count} rows with invalid keys, labels, splits, or dates'

total_rows = features_df.count()
total_positives = features_df.agg(F.sum(TARGET_COL).alias('n')).first()['n']
assert total_rows > 0 and total_positives > 0

print('Total rows:', total_rows)
print('Total positives:', total_positives)
display(features_df.limit(10))

## 3. Build transitive overlap groups

Within each employee-account pair, windows are ordered by start date. A new group begins only when the next start date is later than the maximum end date of every preceding window. Because the intervals are inclusive, equal boundary dates count as overlap.

In [ ]:
order_cols = [WINDOW_START_COL, WINDOW_END_COL, SAMPLE_DATE_COL]
previous_rows = (
    Window.partitionBy(EMPLOYEE_COL, ACCOUNT_COL)
    .orderBy(*order_cols)
    .rowsBetween(Window.unboundedPreceding, -1)
)
cumulative_rows = (
    Window.partitionBy(EMPLOYEE_COL, ACCOUNT_COL)
    .orderBy(*order_cols)
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

features_grouped = (
    features_df
    .withColumn('_previous_max_end', F.max(WINDOW_END_COL).over(previous_rows))
    .withColumn(
        '_new_overlap_group',
        F.when(F.col('_previous_max_end').isNull(), 1)
        .when(F.col(WINDOW_START_COL) > F.col('_previous_max_end'), 1)
        .otherwise(0),
    )
    .withColumn(
        'overlap_component',
        F.sum('_new_overlap_group').over(cumulative_rows).cast('long'),
    )
    .withColumn(
        OVERLAP_GROUP_COL,
        F.sha2(
            F.concat_ws(
                '||',
                F.col(EMPLOYEE_COL),
                F.col(ACCOUNT_COL),
                F.col('overlap_component').cast('string'),
            ),
            256,
        ),
    )
    .drop('_previous_max_end', '_new_overlap_group')
    .cache()
)

overlap_group_summary_spk = (
    features_grouped
    .groupBy(OVERLAP_GROUP_COL)
    .agg(
        F.first(EMPLOYEE_COL).alias(EMPLOYEE_COL),
        F.first(ACCOUNT_COL).alias(ACCOUNT_COL),
        F.min(WINDOW_START_COL).alias('group_window_start'),
        F.max(WINDOW_END_COL).alias('group_window_end'),
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
    )
    .withColumn('negative_count', F.col('row_count') - F.col('positive_count'))
)

print('Number of overlap groups:', overlap_group_summary_spk.count())
display(overlap_group_summary_spk.orderBy(F.desc('row_count')).limit(20))

## 4. Recover the original split targets and current overlap leakage

The original split supplies the actual target row counts/shares, positive counts/shares, and positive rates for train, validation, and test. These are not replaced by assumed percentages. Groups already contained within one split remain fixed; only groups currently crossing splits are candidates for repair.

In [ ]:
original_split_summary = (
    features_grouped
    .groupBy(ORIGINAL_SPLIT_COL)
    .agg(
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
        F.avg(TARGET_COL).alias('positive_rate'),
        F.countDistinct(OVERLAP_GROUP_COL).alias('group_count'),
    )
    .withColumn('row_share', F.col('row_count') / F.lit(total_rows))
    .withColumn('positive_share', F.col('positive_count') / F.lit(total_positives))
)

current_group_split_check = (
    features_grouped
    .groupBy(OVERLAP_GROUP_COL)
    .agg(
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
        F.countDistinct(ORIGINAL_SPLIT_COL).alias('number_of_splits'),
    )
)

current_leakage_summary = current_group_split_check.agg(
    F.count('*').alias('total_overlap_groups'),
    F.sum((F.col('number_of_splits') > 1).cast('long')).alias('groups_crossing_splits'),
    F.sum(F.when(F.col('number_of_splits') > 1, F.col('row_count')).otherwise(0)).alias('rows_in_crossing_groups'),
    F.sum(F.when(F.col('number_of_splits') > 1, F.col('positive_count')).otherwise(0)).alias('positives_in_crossing_groups'),
)

display(original_split_summary)
display(current_leakage_summary)

## 5. Repair only crossing overlap groups

Non-crossing groups keep their original split. For each crossing group, the algorithm evaluates assigning the complete group to train, validation, or test. Candidate loss is calculated over all three splits, not only the candidate split. It combines normalized row-count error, positive-count error, moved-row cost, and overshoot. Groups are ordered by size with randomized tie-breaking, never by positive count. Multiple attempts are run and the complete allocation with the lowest final loss is retained.

```text
loss = ROW_WEIGHT × Σ normalized row error
     + POSITIVE_WEIGHT × Σ normalized positive-count error
     + MOVE_WEIGHT × moved rows / crossing rows
     + OVERSHOOT_WEIGHT × Σ normalized overshoot
```

In [ ]:
original_target_rows = (
    original_split_summary
    .filter(F.col(ORIGINAL_SPLIT_COL).isin(SPLIT_ORDER))
    .select(
        ORIGINAL_SPLIT_COL, 'row_count', 'row_share',
        'positive_count', 'positive_share', 'positive_rate',
    )
    .collect()
)
original_targets = {
    row[ORIGINAL_SPLIT_COL]: {
        'rows': int(row['row_count']),
        'row_share': float(row['row_share']),
        'positives': int(row['positive_count']),
        'positive_share': float(row['positive_share']),
        'positive_rate': float(row['positive_rate']),
    }
    for row in original_target_rows
}
assert set(original_targets) == set(SPLIT_ORDER), f'Unexpected target splits: {original_targets}'

print('Original split targets:')
for split in SPLIT_ORDER:
    print(split, original_targets[split])

crossing_group_ids = (
    current_group_split_check
    .filter(F.col('number_of_splits') > 1)
    .select(OVERLAP_GROUP_COL)
    .cache()
)

# Groups with no leakage keep their original split exactly.
non_crossing_assignment_spk = (
    features_grouped
    .join(crossing_group_ids, on=OVERLAP_GROUP_COL, how='left_anti')
    .select(
        OVERLAP_GROUP_COL,
        F.col(ORIGINAL_SPLIT_COL).alias(NEW_SPLIT_COL),
    )
    .distinct()
)

fixed_summary_rows = (
    features_grouped
    .join(crossing_group_ids, on=OVERLAP_GROUP_COL, how='left_anti')
    .groupBy(ORIGINAL_SPLIT_COL)
    .agg(
        F.count('*').alias('rows'),
        F.sum(TARGET_COL).alias('positives'),
    )
    .collect()
)
fixed_totals = {split: {'rows': 0.0, 'positives': 0.0} for split in SPLIT_ORDER}
for row in fixed_summary_rows:
    fixed_totals[row[ORIGINAL_SPLIT_COL]] = {
        'rows': float(row['rows']),
        'positives': float(row['positives']),
    }

crossing_group_summary_spk = (
    features_grouped
    .join(crossing_group_ids, on=OVERLAP_GROUP_COL, how='inner')
    .groupBy(OVERLAP_GROUP_COL)
    .agg(
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
        *[
            F.sum(
                F.when(F.col(ORIGINAL_SPLIT_COL) == split, 1).otherwise(0)
            ).alias(f'original_{split}_rows')
            for split in SPLIT_ORDER
        ],
    )
)
crossing_group_pd = crossing_group_summary_spk.toPandas()

integer_cols = ['row_count', 'positive_count'] + [
    f'original_{split}_rows' for split in SPLIT_ORDER
]
for column in integer_cols:
    crossing_group_pd[column] = crossing_group_pd[column].astype(np.int64)

total_crossing_rows = int(crossing_group_pd['row_count'].sum())
print('Crossing groups to repair:', len(crossing_group_pd))
print('Rows in crossing groups:', total_crossing_rows)
display(crossing_group_pd.head(10))

In [ ]:
def normalized_absolute_error(actual, target):
    return abs(actual - target) / max(target, EPSILON)


def normalized_overshoot(actual, target):
    return max(actual - target, 0.0) / max(target, EPSILON)


def allocation_objective(current_totals, moved_rows):
    row_error = 0.0
    positive_error = 0.0
    overshoot_error = 0.0

    for split in SPLIT_ORDER:
        target = original_targets[split]
        actual = current_totals[split]
        row_error += normalized_absolute_error(actual['rows'], target['rows'])
        positive_error += normalized_absolute_error(actual['positives'], target['positives'])
        overshoot_error += normalized_overshoot(actual['rows'], target['rows'])
        overshoot_error += normalized_overshoot(actual['positives'], target['positives'])

    move_error = moved_rows / max(total_crossing_rows, 1)
    return (
        ROW_WEIGHT * row_error
        + POSITIVE_WEIGHT * positive_error
        + MOVE_WEIGHT * move_error
        + OVERSHOOT_WEIGHT * overshoot_error
    )


def allocate_crossing_groups(crossing_groups, random_seed):
    work = crossing_groups.copy()
    rng = np.random.default_rng(random_seed)
    work['_tie_breaker'] = rng.random(len(work))

    # Large groups are harder to place, but label is never used for ordering.
    work = work.sort_values(
        ['row_count', '_tie_breaker'],
        ascending=[False, True],
    ).reset_index(drop=True)

    current = {
        split: {
            'rows': fixed_totals[split]['rows'],
            'positives': fixed_totals[split]['positives'],
        }
        for split in SPLIT_ORDER
    }
    moved_rows = 0.0
    assignments = []

    for row in work.itertuples(index=False):
        candidate_splits = list(SPLIT_ORDER)
        rng.shuffle(candidate_splits)
        best_split = None
        best_score = np.inf
        best_moved_rows = None

        for split in candidate_splits:
            projected = {name: values.copy() for name, values in current.items()}
            projected[split]['rows'] += row.row_count
            projected[split]['positives'] += row.positive_count

            group_moved_rows = row.row_count - getattr(row, f'original_{split}_rows')
            projected_moved_rows = moved_rows + group_moved_rows
            score = allocation_objective(projected, projected_moved_rows)

            if score < best_score:
                best_split = split
                best_score = score
                best_moved_rows = projected_moved_rows

        assignments.append(best_split)
        current[best_split]['rows'] += row.row_count
        current[best_split]['positives'] += row.positive_count
        moved_rows = best_moved_rows

    work[NEW_SPLIT_COL] = assignments
    final_objective = allocation_objective(current, moved_rows)
    return work.drop(columns=['_tie_breaker']), current, moved_rows, final_objective


best_crossing_assignment = None
best_current_totals = None
best_moved_rows = None
best_objective = np.inf
attempt_results = []

for attempt in range(N_ATTEMPTS):
    attempt_seed = RANDOM_SEED + attempt
    assignment, current_totals, moved_rows, objective = allocate_crossing_groups(
        crossing_group_pd, attempt_seed
    )
    attempt_results.append({
        'attempt': attempt + 1,
        'random_seed': attempt_seed,
        'objective': objective,
        'moved_rows': moved_rows,
    })

    if objective < best_objective:
        best_crossing_assignment = assignment
        best_current_totals = current_totals
        best_moved_rows = moved_rows
        best_objective = objective

assert best_crossing_assignment is not None, 'N_ATTEMPTS must be at least 1'
assignment_records = [
    (str(group_id), str(split))
    for group_id, split in best_crossing_assignment[
        [OVERLAP_GROUP_COL, NEW_SPLIT_COL]
    ].itertuples(index=False, name=None)
]
crossing_assignment_spk = spark.createDataFrame(
    assignment_records,
    schema=f'{OVERLAP_GROUP_COL} string, {NEW_SPLIT_COL} string',
)
group_assignment_spk = (
    non_crossing_assignment_spk
    .unionByName(crossing_assignment_spk)
    .cache()
)

attempt_results_pd = pd.DataFrame(attempt_results).sort_values('objective')
display(attempt_results_pd)
print('Best objective:', best_objective)
print('Moved rows:', best_moved_rows)
print('Final totals:', best_current_totals)
display(group_assignment_spk.groupBy(NEW_SPLIT_COL).count())

## 6. Join assignments back and validate the hard constraint

In [ ]:
features_with_new_split = features_grouped.join(
    group_assignment_spk, on=OVERLAP_GROUP_COL, how='left'
)

group_integrity = (
    features_with_new_split
    .groupBy(OVERLAP_GROUP_COL)
    .agg(F.countDistinct(NEW_SPLIT_COL).alias('number_of_new_splits'))
)
missing_assignments = features_with_new_split.filter(F.col(NEW_SPLIT_COL).isNull()).count()
groups_crossing_new_splits = group_integrity.filter(F.col('number_of_new_splits') != 1).count()
actual_moved_rows = features_with_new_split.filter(
    F.col(ORIGINAL_SPLIT_COL) != F.col(NEW_SPLIT_COL)
).count()
non_crossing_rows_changed = (
    features_with_new_split
    .join(crossing_group_ids, on=OVERLAP_GROUP_COL, how='left_anti')
    .filter(F.col(ORIGINAL_SPLIT_COL) != F.col(NEW_SPLIT_COL))
    .count()
)

# Adjacent overlap components from the same pair must not overlap each other.
component_ranges = (
    features_grouped
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL, 'overlap_component')
    .agg(
        F.min(WINDOW_START_COL).alias('component_start'),
        F.max(WINDOW_END_COL).alias('component_end'),
    )
)
component_order = Window.partitionBy(EMPLOYEE_COL, ACCOUNT_COL).orderBy('overlap_component')
overlapping_separate_components = (
    component_ranges
    .withColumn('previous_component_end', F.lag('component_end').over(component_order))
    .filter(F.col('component_start') <= F.col('previous_component_end'))
    .count()
)

hard_constraint_passed = (
    missing_assignments == 0
    and groups_crossing_new_splits == 0
    and overlapping_separate_components == 0
    and non_crossing_rows_changed == 0
    and actual_moved_rows == int(best_moved_rows)
)
hard_constraint_summary = pd.DataFrame([{
    'missing_assignments': missing_assignments,
    'groups_crossing_new_splits': groups_crossing_new_splits,
    'overlapping_separate_components': overlapping_separate_components,
    'non_crossing_rows_changed': non_crossing_rows_changed,
    'actual_moved_rows': actual_moved_rows,
    'hard_constraint_passed': hard_constraint_passed,
}])
display(hard_constraint_summary)
assert hard_constraint_passed, 'Issue 3B overlap-group constraint failed'

## 7. Compare original and new label balance

In [ ]:
new_split_summary = (
    features_with_new_split
    .groupBy(NEW_SPLIT_COL)
    .agg(
        F.count('*').alias('new_row_count'),
        F.sum(TARGET_COL).alias('new_positive_count'),
        F.avg(TARGET_COL).alias('new_positive_rate'),
        F.countDistinct(OVERLAP_GROUP_COL).alias('new_group_count'),
    )
    .withColumn('new_row_share', F.col('new_row_count') / F.lit(total_rows))
    .withColumn('new_positive_share', F.col('new_positive_count') / F.lit(total_positives))
)

comparison_summary = (
    original_split_summary
    .select(
        F.col(ORIGINAL_SPLIT_COL).alias('split'),
        F.col('row_count').alias('original_row_count'),
        F.col('row_share').alias('original_row_share'),
        F.col('positive_count').alias('original_positive_count'),
        F.col('positive_share').alias('original_positive_share'),
        F.col('positive_rate').alias('original_positive_rate'),
    )
    .join(
        new_split_summary.select(
            F.col(NEW_SPLIT_COL).alias('split'),
            'new_row_count', 'new_row_share', 'new_positive_count',
            'new_positive_share', 'new_positive_rate', 'new_group_count',
        ),
        on='split',
    )
    .withColumn('row_share_difference', F.col('new_row_share') - F.col('original_row_share'))
    .withColumn('positive_share_difference', F.col('new_positive_share') - F.col('original_positive_share'))
    .withColumn('positive_rate_difference', F.col('new_positive_rate') - F.col('original_positive_rate'))
)
display(comparison_summary)

## 8. Simple feasibility decision

The thresholds match the issue 3A notebook and can be changed according to the project requirements.

In [ ]:
MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE = 0.02
MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE = 0.02
MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE = 0.10
MIN_POSITIVES_IN_VAL_OR_TEST = 100

comparison_pd = comparison_summary.toPandas()
comparison_pd['absolute_row_share_difference'] = comparison_pd['row_share_difference'].abs()
comparison_pd['absolute_positive_share_difference'] = comparison_pd['positive_share_difference'].abs()
comparison_pd['relative_positive_rate_difference'] = (
    comparison_pd['positive_rate_difference'].abs()
    / comparison_pd['original_positive_rate'].replace(0, np.nan)
)
comparison_pd['row_balance_passed'] = comparison_pd['absolute_row_share_difference'] <= MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE
comparison_pd['positive_share_balance_passed'] = comparison_pd['absolute_positive_share_difference'] <= MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE
comparison_pd['positive_rate_balance_passed'] = comparison_pd['relative_positive_rate_difference'] <= MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE
comparison_pd['minimum_positive_count_passed'] = True
comparison_pd.loc[
    comparison_pd['split'].isin(['val', 'test']),
    'minimum_positive_count_passed',
] = comparison_pd.loc[
    comparison_pd['split'].isin(['val', 'test']), 'new_positive_count'
] >= MIN_POSITIVES_IN_VAL_OR_TEST

all_balance_checks_passed = bool(comparison_pd[[
    'row_balance_passed', 'positive_share_balance_passed',
    'positive_rate_balance_passed', 'minimum_positive_count_passed',
]].all().all())
overall_feasible = bool(hard_constraint_passed and all_balance_checks_passed)

display(comparison_pd)
print('Hard constraint passed:', hard_constraint_passed)
print('All balance checks passed:', all_balance_checks_passed)
print('Overall issue 3B split considered feasible:', overall_feasible)

## 9. Build the complete feature table and recreate `cv_fold`

The original Delta feature table is read again with all columns. The repaired split replaces `label_split`. All rows in the new training split are then reassigned to five label-stratified random CV folds. Validation and test rows receive a null `cv_fold`. The final table keeps exactly the original column names and does not add the old split or audit columns.

In [ ]:
assert overall_feasible, 'The repaired split did not pass validation'

full_feature_source = (
    spark.read
    .format('delta')
    .load(FEATURE_TABLE_PATH)
    .cache()
)
source_row_count = full_feature_source.count()
assert source_row_count == total_rows, (
    f'Full feature source has {source_row_count} rows; split analysis has {total_rows}'
)

mapping_keys = [
    EMPLOYEE_COL, ACCOUNT_COL, SAMPLE_DATE_COL,
    WINDOW_START_COL, WINDOW_END_COL,
]
split_mapping = (
    features_with_new_split
    .select(*mapping_keys, NEW_SPLIT_COL)
    .dropDuplicates(mapping_keys)
    .cache()
)
mapping_row_count = split_mapping.count()
assert mapping_row_count == total_rows, (
    f'Split mapping has {mapping_row_count} unique keys; expected {total_rows}'
)

source_alias = full_feature_source.alias('source')
mapping_alias = split_mapping.alias('mapping')
join_conditions = [
    F.upper(F.col(f'source.{EMPLOYEE_COL}')) == F.col(f'mapping.{EMPLOYEE_COL}'),
    F.col(f'source.{ACCOUNT_COL}').cast('string') == F.col(f'mapping.{ACCOUNT_COL}'),
    F.to_date(F.col(f'source.{SAMPLE_DATE_COL}')) == F.col(f'mapping.{SAMPLE_DATE_COL}'),
    F.to_date(F.col(f'source.{WINDOW_START_COL}')) == F.col(f'mapping.{WINDOW_START_COL}'),
    F.to_date(F.col(f'source.{WINDOW_END_COL}')) == F.col(f'mapping.{WINDOW_END_COL}'),
]

columns_to_keep = [
    column for column in full_feature_source.columns
    if column not in [ORIGINAL_SPLIT_COL, 'cv_fold']
]
complete_features_base = (
    source_alias
    .join(mapping_alias, on=join_conditions, how='left')
    .select(
        *[F.col(f'source.`{column}`').alias(column) for column in columns_to_keep],
        F.col(f'mapping.{NEW_SPLIT_COL}').alias(ORIGINAL_SPLIT_COL),
    )
    .cache()
)

joined_row_count = complete_features_base.count()
missing_split_count = complete_features_base.filter(
    F.col(ORIGINAL_SPLIT_COL).isNull()
).count()
assert joined_row_count == total_rows, f'Join produced {joined_row_count}; expected {total_rows}'
assert missing_split_count == 0, f'{missing_split_count} rows did not receive the repaired split'

train_fold_window = (
    Window.partitionBy(TARGET_COL)
    .orderBy(F.rand(seed=CV_FOLD_SEED))
)
new_training_rows = (
    complete_features_base
    .filter(F.col(ORIGINAL_SPLIT_COL) == 'train')
    .withColumn('_fold_rank', F.row_number().over(train_fold_window))
    .withColumn(
        'cv_fold',
        ((F.col('_fold_rank') - 1) % N_CV_FOLDS).cast('int'),
    )
    .drop('_fold_rank')
)
new_validation_and_test_rows = (
    complete_features_base
    .filter(F.col(ORIGINAL_SPLIT_COL) != 'train')
    .withColumn('cv_fold', F.lit(None).cast('int'))
)

final_feature_table = (
    new_training_rows
    .unionByName(new_validation_and_test_rows)
    .select(*full_feature_source.columns)
    .cache()
)

invalid_cv_fold_count = final_feature_table.filter(
    (
        (F.col(ORIGINAL_SPLIT_COL) == 'train')
        & (F.col('cv_fold').isNull() | ~F.col('cv_fold').between(0, N_CV_FOLDS - 1))
    )
    | (
        (F.col(ORIGINAL_SPLIT_COL) != 'train')
        & F.col('cv_fold').isNotNull()
    )
).count()
assert final_feature_table.count() == source_row_count
assert final_feature_table.columns == full_feature_source.columns
assert invalid_cv_fold_count == 0, f'Found {invalid_cv_fold_count} rows with invalid cv_fold'

if 'insider_label' in final_feature_table.columns:
    insiders_outside_test = final_feature_table.filter(
        (F.col('insider_label') == 1)
        & (F.col(ORIGINAL_SPLIT_COL) != 'test')
    ).count()
    assert insiders_outside_test == 0, (
        f'{insiders_outside_test} insider_label rows are outside test'
    )

display(
    final_feature_table
    .groupBy(ORIGINAL_SPLIT_COL, 'cv_fold', TARGET_COL)
    .count()
    .orderBy(ORIGINAL_SPLIT_COL, 'cv_fold', TARGET_COL)
)

## 10. Save the complete feature table

This writes a new Delta dataset. The original feature table is not overwritten. The saved schema matches the original feature table, while `label_split` and `cv_fold` contain the repaired values.

In [ ]:
(
    final_feature_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(FULL_FEATURE_OUTPUT_PATH)
)
print('Saved complete feature table:', FULL_FEATURE_OUTPUT_PATH)

## Interpretation

Issue 3B is feasible when the overlap hard constraint passes and the repaired split remains within the configured balance thresholds. Non-crossing groups remain unchanged, so differences from the original allocation come only from rows in crossing groups. Compared with issue 3A, non-overlapping time periods from the same employee-account pair may still appear in different splits, providing more allocation flexibility but weaker identity-level isolation.